# 🧠 SOMA: Adjudication Engine (Module 3)
## **Module 3: Neural Ingestion & Cognitive Analysis**

---

### **Overview**
This notebook serves as the primary intelligence layer of the SOMA project. Rather than performing simple keyword filtering, this engine applies **near-cognitive analysis** to evaluate the subtext, intent, and "cognitive friction" of linguistic compounds.

### **The Hybrid Pipeline**
We are ingesting data from the **SOMA Data Factory**, which provides a balanced registry of:
1. **Synthetic Lab Samples:** Engineered to test specific "Gray Zone" weaknesses.
2. **Real-World Samples:** Organic human communication to ensure real-world grounding.

## Wake up the Memory Core!

In [2]:
import sys
import os

# Tell Python to look inside the 'src' folder
sys.path.append(os.path.abspath('src'))

# Now wake up the Memory Core
try:
    from logger import SomaLogger
    memory = SomaLogger()
    print("SOMA Memory Core successfully linked from /src.")
except ModuleNotFoundError:
    print("Error: Still can't find 'logger.py'. Check if the folder name is exactly 'src'.")

SOMA Memory Core successfully linked from /src.


## Imports

In [2]:
%%capture
# Your code here
import vertexai
from vertexai.generative_models import GenerativeModel, GenerationConfig
import yaml
import json
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
import json
import logging
import html
import re
from typing import Dict, List, Any, Optional
import time
import warnings

# This tells Python to ignore all non-critical alerts
warnings.filterwarnings('ignore')

## Environment Initialization

In [4]:
def initialize_soma_engine(project_id="project-b75ad945-ea4d-46ac-977", location="us-central1"):
    """
    Initializes the Vertex AI environment and the Gemini model instance.
    
    Args:
        project_id (str): Your Google Cloud Project ID.
        location (str): The GCP region for the Vertex AI resources.
        
    Returns:
        GenerativeModel: The initialized AI model instance.
    """
    vertexai.init(project=project_id, location=location)
    model_name = "gemini-2.5-flash"
    
    model_instance = GenerativeModel(
        model_name,
        generation_config=GenerationConfig(
            temperature=0.3,
            response_mime_type="application/json"
        )
    )
    
    print(f" SOMA Engine Initialized: {model_name}")
    return model_instance

model = initialize_soma_engine(project_id="project-b75ad945-ea4d-46ac-977")

 SOMA Engine Initialized: gemini-2.5-flash


## Sensory Intake (The Data Handler)

In [5]:

# Configure logging for the sensory intake
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class DataHandler:
    """
    The intake gateway for the SOMA moderation system or the "sensory intake".
    Handles retrieval, sanitization, and verification of incoming data.
    """

    def __init__(self, source_config: Optional[Dict[str, Any]] = None):
        """Initializes the data handler with necessary settings."""
        self.source_config = source_config or {}

    def ingest_registry(self, file_path: str) -> List[Dict[str, Any]]:
        """
        Ingests a SOMA JSONL registry and repairs malformed entries.
        Resolves the structural inconsistencies found in the batch.
        """
        sanitized_compounds = []
        failures = 0

        try:
            # 'utf-8-sig' handles invisible Byte Order Marks
            with open(file_path, 'r', encoding='utf-8-sig') as f:
                for line_num, line in enumerate(f, 1):
                    clean_line = line.strip()
                    if not clean_line:
                        continue
                    
                    try:
                        # Stage 1: Standard JSON decoding
                        data = json.loads(clean_line)
                        
                        # Stage 2: Normalization for string-wrapped JSON
                        if isinstance(data, str):
                            data = json.loads(data)
                        
                        # Stage 3: Sensory Sanitization
                        clean_data = self.sanitize(data)
                        
                        # Stage 4: Gatekeeper Validation
                        if self.validate_schema(clean_data):
                            sanitized_compounds.append(clean_data)
                        else:
                            failures += 1

                    except json.JSONDecodeError:
                        # Stage 5: Resilient Healing for string-wrapped JSON
                        try:
                            repaired = clean_line.strip('"').replace('\\"', '"')
                            healed_data = self.sanitize(json.loads(repaired))
                            if self.validate_schema(healed_data):
                                sanitized_compounds.append(healed_data)
                            else:
                                failures += 1
                        except:
                            failures += 1
                            continue

            logging.info(f"Ingestion Complete: {len(sanitized_compounds)} items loaded. {failures} rejected.")
            return sanitized_compounds

        except FileNotFoundError:
            logging.error(f"Critical Error: File {file_path} not found.")
            return []

    def sanitize(self, data: Dict[str, Any]) -> Dict[str, Any]:
        """Cleans raw input and normalizes keys for the SOMA Engine."""
        # Ensure 'text' is mapped to 'raw_text' for the Adjudication Cycle
        text_key = "raw_text" if "raw_text" in data else "text"
        text = data.get(text_key, "")
        
        # Strip HTML and unescape characters
        clean_text_pattern = re.compile('<.*?>')
        text = re.sub(clean_text_pattern, '', str(text))
        text = html.unescape(text)
        
        data["raw_text"] = text.strip()
        return data

    def validate_schema(self, data: Dict[str, Any]) -> bool:
        """Verifies data structure matches the Fail-Fast Principle."""
        if not isinstance(data, dict):
            return False
        if "raw_text" not in data or not str(data["raw_text"]).strip():
            return False
        return True

# --- EXECUTION ---
handler = DataHandler()
target_file = 'soma_registry_batch_20260511_1419.jsonl'

# This call replaces your old load_compounds calls
compounds = handler.ingest_registry(target_file)

## Diagnostic Check (Metacognition)

In [6]:
# --- SOMA Diagnostic: Registry Integrity Check ---
# Theory: This cell acts as a Metacognitive Filter. It verifies that the 
# 'Sensory Intake' stage has provided high-fidelity data before we 
# commit API resources to analysis. (Fail-Fast)

if 'compounds' in globals() and len(compounds) > 0:
    print(f"--- SOMA Registry Diagnostic ---")
    print(f"Total Compounds Ingested: {len(compounds)}")
    
    # 1. Check for Origin Balance
    # Ensures a healthy mix of Synthetic Lab Samples and Real-World Samples
    origins = [c.get('origin', 'unknown') for c in compounds]
    from collections import Counter
    print(f"Distribution: {Counter(origins)}")
    
    # 2. Key Verification
    # Confirms the DataHandler mapped all text to the 'raw_text' key
    missing_keys = [i for i, c in enumerate(compounds) if 'raw_text' not in c]
    if missing_keys:
        print(f"Warning: {len(missing_keys)} items are missing 'raw_text'.")
    else:
        print("Key Verification: All items are model-ready.")
        
    # 3. Sensory Peek
    # A random glance at a record to ensure sanitization (HTML stripping) worked
    import random
    sample = random.choice(compounds)
    print(f"\n--- Random Sample Entry ---")
    print(f"Origin: {sample.get('origin')}")
    print(f"Content: {sample.get('raw_text')[:100]}...")
    
else:
    print("Diagnostic Failed: No data found in 'compounds'.")

--- SOMA Registry Diagnostic ---
Total Compounds Ingested: 125
Distribution: Counter({'real_world_dataset': 75, 'Real_World_Ingestion': 50})
Key Verification: All items are model-ready.

--- Random Sample Entry ---
Origin: Real_World_Ingestion
Content: @Doop_3 lol shid who got one? Boy my ass lonely as a bitch. You cuffed up ain't ya...


## Policy Ingestion (Executive Function)

In [7]:
def load_soma_policy(file_path='configs/policy.yml'):
    """
    Loads the safety and adjudication standards from the configs directory.
    """
    try:
        with open(file_path, 'r') as f:
            policy_data = yaml.safe_load(f)
        policy_string = yaml.dump(policy_data, default_flow_style=False)
        print(f"Status: Policy loaded successfully from {file_path}")
        return policy_string
    except Exception as e:
        print(f"Error loading policy: {e}")
        # Provides a fallback if the path is still unreachable
        return "Analyze for intent clarity and cognitive friction."

current_policy = load_soma_policy()

Status: Policy loaded successfully from configs/policy.yml


## Define and Analyze

In [ ]:
import json
import time

# 1. THE ANALYZER (No changes needed, but keeping it here for consistency)
def analyze_cognitive_intent(content, model, policy):
    if not content or str(content).strip() == "":
        return {"verdict": "ERROR", "rationale": "CONTENT_WAS_EMPTY"}
        
    prompt = f"Policy: {policy}\n\nContent: {content}\n\nAnalyze this content. Provide a verdict (ALLOW/BLOCK) and a rationale in JSON format."
    try:
        response = model.generate_content(prompt)
        raw_text = response.text
        try:
            clean_json = raw_text.replace("```json", "").replace("```", "").strip()
            data = json.loads(clean_json)
            return {
                "verdict": data.get("verdict", "ALLOW"),
                "rationale": data.get("rationale", "No rationale provided."),
                "raw_response": raw_text
            }
        except:
            return {
                "verdict": "BLOCK" if "BLOCK" in raw_text.upper() else "ALLOW",
                "rationale": raw_text[:250],
                "raw_response": raw_text
            }
    except Exception as e:
        return {"verdict": "ERROR", "rationale": f"API Error: {str(e)}"}

# 2. THE CORRECTED EXECUTION LOOP (Now looking for 'raw_text')
def run_soma_batch(data_list, model_obj, policy_obj, memory_obj):
    print(f"Starting Adjudication for {len(data_list)} items...")
    
    for i, entry in enumerate(data_list):
        # FIX: Added 'raw_text' as the primary source key
        content = entry.get('raw_text') or entry.get('text') or entry.get('body') or ""
        
        if not content:
            print(f"Skipping Item {i+1}: Still no text found in keys {list(entry.keys())}")
            continue

        # Get Analysis
        analysis = analyze_cognitive_intent(content, model_obj, policy_obj)
        
        # Log to the Audit Trail
        memory_obj.log_decision(
            module_name="Adjudicator", 
            decision=analysis.get("verdict"), 
            rationale=analysis.get("rationale"), 
            cognitive_system="System 2"
        )
        
        # Visual Progress
        print(f"[{i+1}/{len(data_list)}] Success! Rationale: {analysis.get('rationale')[:60]}...", flush=True)
        
        # Pause to avoid API rate limits (15 RPM)
        time.sleep(4)

# 3. RUN IT
run_soma_batch(compounds, model, current_policy, memory)

Starting Adjudication for 125 items...
[1/125] Success! Rationale: The content contains a derogatory term ("bitch") directed at...
[2/125] Success! Rationale: The content contains a highly aggressive, sexually explicit,...
[3/125] Success! Rationale: The content contains the derogatory and dehumanizing term 'h...
[4/125] Success! Rationale: The content exhibits high systemic toxicity and dehumanizing...
[5/125] Success! Rationale: The content contains highly explicit and vulgar sexual langu...


In [ ]:
# DIAGNOSTIC TEST: Run this and paste the result
print("--- 1. Testing analyze_cognitive_intent directly ---")
sample_analysis = analyze_cognitive_intent("This is a test message.", model, current_policy)
print(f"Analysis Dictionary: {sample_analysis}")

print("\n--- 2. Testing Variable Scope ---")
# Check if 'rationale' exists in the global space
if 'rationale' in globals():
    print(f"GLOBAL CONFLICT FOUND: The variable 'rationale' is globally set to: {globals()['rationale']}")
else:
    print("No global 'rationale' conflict.")

print("\n--- 3. Testing Logger behavior ---")
# Manually call the logger with a unique string
test_string = "UNIQUE_LOG_TEST_12345"
memory.log_decision("Diag_Test", "ALLOW", test_string, "System 2")
print("Check your log file for 'UNIQUE_LOG_TEST_12345'. Does it show that, or still 'Batch'?")

In [ ]:
# Use 'memory' here because that's what we initialized in Step 1
test_data = [{"text": "Testing the system rationale extraction."}]
adjudicated_test = execute_adjudication_cycle(test_data, model, current_policy, memory)

In [ ]:
import json
import time

# 1. THE DATA SENTINEL (Let's see what is actually inside your compounds)
if 'compounds' in globals() and len(compounds) > 0:
    print(f"--- DATA CHECK ---")
    print(f"Keys available in your data: {list(compounds[0].keys())}")
    # We will prioritize 'text', then 'body', then 'content'
else:
    print("ERROR: 'compounds' is still not defined or is empty.")

# 2. THE ANALYZER (Handles the "No Content" issue)
def analyze_cognitive_intent(content, model, policy):
    if not content or str(content).strip() == "":
        return {"verdict": "ERROR", "rationale": "CONTENT_WAS_EMPTY_BEFORE_SENDING"}
        
    prompt = f"Policy: {policy}\n\nContent: {content}\n\nAnalyze this content. Provide a verdict (ALLOW/BLOCK) and a rationale in JSON format."
    try:
        response = model.generate_content(prompt)
        raw_text = response.text
        try:
            clean_json = raw_text.replace("```json", "").replace("```", "").strip()
            data = json.loads(clean_json)
            return {
                "verdict": data.get("verdict", "ALLOW"),
                "rationale": data.get("rationale", "No rationale provided."),
                "raw_response": raw_text
            }
        except:
            return {
                "verdict": "BLOCK" if "BLOCK" in raw_text.upper() else "ALLOW",
                "rationale": raw_text[:250],
                "raw_response": raw_text
            }
    except Exception as e:
        return {"verdict": "ERROR", "rationale": f"API Error: {str(e)}"}

# 3. THE RE-ENGINEERED EXECUTION LOOP
def run_soma_batch(data_list, model_obj, policy_obj, memory_obj):
    print(f"Starting Adjudication for {len(data_list)} items...")
    
    for i, entry in enumerate(data_list):
        # We try every common key to find the actual text
        content = entry.get('text') or entry.get('body') or entry.get('content') or ""
        
        # If still empty, skip the API call and log the error
        if not content:
            print(f"Skipping Item {i+1}: No text found in keys {list(entry.keys())}")
            continue

        # Get Analysis
        analysis = analyze_cognitive_intent(content, model_obj, policy_obj)
        
        # Log to the Audit Trail
        memory_obj.log_decision(
            module_name="Adjudicator", 
            decision=analysis.get("verdict"), 
            rationale=analysis.get("rationale"), 
            cognitive_system="System 2"
        )
        
        print(f"[{i+1}] Processed. Rationale: {analysis.get('rationale')[:50]}...", flush=True)
        time.sleep(4)

# 4. RUN IT
run_soma_batch(compounds, model, current_policy, memory)

## Execute

## Stage 3: Batch Adjudication & Audit Log Persistence

In [ ]:
# Adjucation Processing Confirmation 
import time
import sys

if compounds:
    total = len(compounds)
    print(f"SOMA Adjudication: Processing {total} samples...")
    
    results = []
    # Simple spinner frames
    spinner = ['|', '/', '-', '\\']
    
    for i, entry in enumerate(compounds):
        # 1. THE ACTION (Silent)
        analysis = analyze_cognitive_intent(entry.get('text', ''), model, current_policy)
        
        # 2. THE MEMORY (Background)
        memory.log_decision("Adjudicator", analysis.get("verdict"), "Batch", "System 2")
        results.append({**entry, "soma_analysis": analysis})
        
        # 3. THE QUIET STATUS
        # This updates only the number and the spinner on ONE line
        symbol = spinner[i % len(spinner)]
        sys.stdout.write(f"\r{symbol} Progress: {i+1}/{total} samples committed to log...")
        sys.stdout.flush()
        
        # 4. RATE LIMIT SAFETY
        time.sleep(4)

    print(f"\n\nDone. Final audit available in: logs/soma_audit.log")
else:
    print("Inference aborted: No compounds found.")

SOMA Adjudication: Processing 125 samples...
/ Progress: 30/125 samples committed to log...

## Final Execution Test

In [ ]:
# SOMA Deep Vision Test
if compounds:
    print(f"Commencing Deep Vision on {len(compounds[:1])} sample...")
    
    for i, entry in enumerate(compounds[:1]):
        content = entry.get('raw_text') or entry.get('text', '')
        
        try:
            # 1. Call the AI
            analysis = analyze_cognitive_intent(content, model, current_policy)
            
            # 2. THE SMOKING GUN: Print everything the AI said
            print("\n--- Raw AI Output Received ---")
            print(analysis) 
            print("------------------------------\n")
            
            # 3. Check the specific key
            verdict = analysis.get('verdict', 'KEY_MISSING')
            print(f"Extracted Verdict: {verdict}")
            
        except Exception as e:
            print(f"Error during deep vision: {e}")
else:
    print("No compounds found.")